# 学习率过大导致训练崩溃

## 现象描述
<font color='red'>Loss 先正常下降 → 突然跳到极大值 → 变为 NaN</font>

<font color='blue'>这是一个典型的训练发散（Divergence）模式，而非简单的震荡：</font>
1. **前期正常下降**：
   - 说明模型初始化、数据流、损失函数等基本设置是正确的，<font color='red'>优化方向大致正确</font>。
2. **突然跳到极大值**：
   - <font color='red'>不是渐进式的恶化，而是断崖式崩溃</font>，**往往发生在某个特定的迭代步**。
3. **变为 NaN**：
   - <font color='blue'>一旦参数进入数值不稳定区域，后续计算（如 log(0)、exp(极大值)、1/0 等）会产生无穷大或非法数值，污染整个网络</font>。
     - 替换 NaN（不删除，改为指定值）： 如果需要在保持张量形状不变的前提下处理 NaN


In [ ]:
# 将 NaN 替换为 0
x_filled = torch.nan_to_num(x, nan=0.0)

# 也可替换 Inf
x_filled = torch.nan_to_num(x, nan=0.0, posinf=1e4, neginf=-1e4)

## 数学机制：为什么学习率过大会导致崩溃？
###  梯度下降的基本更新公式
参数更新：

    $θ_{t+1} = θ_t − η⋅∇_θL(θ_t)$
- 其中 η  是学习率。
  - <font color='red'>当 η  过大时，更新步长 η⋅∇L  可能远超损失函数的"安全更新范围"</font>。    

### 损失曲面的局部几何
想象损失函数 L(θ)  是一个非凸（non-convex）的复杂地形：

In [ ]:
        Loss
         ↑
         |    \       /
         |     \  谷底  /
         |      \   /      ← 正常学习率：稳步下降
         |       \/        ← 过大学习率：一步跨过谷底
         |       /\           冲到对面山坡上
         |      /  \          甚至越过山脊进入深渊
         |     /    \
         +----------------→ θ

- 正常学习率：参数沿着梯度方向，**逐步滑向谷底**（局部最优）。
- 过大学习率：<font color='red'>参数更新步长过大，"跨过"了最优区域，甚至冲到了损失曲面的另一侧更高处。</font>

### 从"极大值"到"NaN"的<font color='red'>连锁反应</font>
**一旦参数被推到损失曲面的"危险区域"，以下数值问题会连锁爆发：**

| 环节                     | 具体机制                                                  |
| ---------------------- | ----------------------------------------------------- |
| **激活值爆炸**              | 在**深层网络**中，<font color='red'>过大的权重导致前向传播时激活值指数级增长（如 `ReLU` 不断累积正值）</font>。         |
| **梯度爆炸**               | 根据链式法则，反向传播时<font color='red'>梯度与激活值相乘</font>，导致梯度越来越大，形成正反馈。                  |
| **Softmax / Log 数值溢出** | 若 logits 极大，`exp(x)` 会溢出为 **`inf`**；`log(0)` 会得到 **`-inf`**。  |
| **交叉熵损失计算**            | <font color='red'>若概率预测为 0 或 1（由于数值溢出），`y·log(p)` 会产生 `NaN`。</font>            |
| **参数变为 NaN / Inf**     | <font color='red'>一旦某层参数变成 `NaN`，后续所有涉及该参数的计算都会传播 `NaN`，**整个模型不可逆地损坏**。</font> |

#### 用公式直观展示：
  - 假设某层输出 z=W⋅x+b ，<font color='red'>若 W  因大学习率更新变得极大</font>
  - 则 z →→→ ∞ 
  - Softmax: $p_i = \frac{e^z_i}{∑_j{e^z_j}}$ →→→  <font color='red'>数值上溢或下溢</font>
  - 交叉熵: $L=−∑y_ilog(p_i)$ →→→ ∞  或 NaN
  
  

## 三、为什么偏偏是"突然"跳变？

<font color='red'>这与损失曲面的曲率**非均匀性**密切相关：</font>
- 前期在"平缓区域"：
  - <font color='blue'>训练初期，参数可能位于损失曲面的相对平缓区域（梯度较小），即使学习率偏大，更新步长也尚可接受，loss 能正常下降。</font>
  
- <font color='blue'>进入"高曲率区域"：</font>
  - <font color='red'>随着优化进行，参数可能接近某个陡峭的峡谷或尖锐的极小值。此时梯度方向变化剧烈（Hessian矩阵的特征值差异大）。</font>
  
- 一步踏错：
  - <font color='blue'>在该点，梯度可能突然变大，或者曲率使得同样的步长在该区域的实际影响被放大。大学习率在此刻相当于"在悬崖边迈了一大步"，直接坠入数值深渊。</font>

<font color='red'>在强化学习中，这一问题更为严重，因为策略梯度（Policy Gradient）本身方差就高，加上环境交互的非平稳性，损失曲面更加崎岖。</font>

## 解决方案详解

### 降低学习率（如从 1e-3 → 1e-4）
- **原理：直接减小步长 η ，确保参数更新在损失函数的局部二次近似有效范围内。**
  - 1e-3 是深度学习常用的<font color='red'>"经验默认值"</font>，<font color='blue'>但对于某些复杂任务（如RL、Transformer训练）可能过大</font>。
  - <font color='blue'>1e-4 或更小（如 3e-5）在以下场景更常见：</font>
    - **强化学习（策略梯度方差大）**
    - **微调大语言模型（LLM Fine-tuning）**
    - **使用较大批次（Batch Size）时（梯度更稳定，但需要更小学习率配合**
    
- **局限：学习率过小会导致：**
  - 收敛极慢
  - 陷入浅层局部最优
  - 在平原区域（梯度极小）几乎无法前进
  
### 学习率预热（Warmup）
- <font color='red'>原理：训练初期从一个极小的学习率开始，逐步增大到目标值，给模型一个"适应期"</font>

In [ ]:
学习率
  ↑
  |         ___________
  |        /           ← 目标学习率（如 1e-3）
  |       /
  |      /
  |     /
  |    /
  |___/________________→ 迭代步数
     ↑
   Warmup阶段（如前1000步线性增长）

#### 为什么 Warmup 有效？

| 阶段       | 问题                                                                | Warmup 的作用         |
| -------- | ----------------------------------------------------------------- | ------------------ |
| **训练初期** | 参数随机初始化，梯度方向极不稳定且可能很大；Adam等自适应优化器的二阶矩估计 $v_t$ 尚未充分积累，导致更新步长被异常放大。 | 小学习率限制步长，防止初期"暴走"。 |
| **预热后期** | 梯度统计量（一阶矩、二阶矩）<font color='red'>趋于稳定</font>，模型已进入合理的参数区域。                                  | 逐步恢复目标学习率，保证收敛速度。  |


<font color='red'>Warmup 的变体</font>：
- **Linear Warmup：**
  - 学习率从 0 线性增长到目标值，持续 N  步。
- **Constant Warmup：**
  - 先固定小学习率一段时间，再跳变到目标值。
- **Cosine Warmup：**
  - 结合 Cosine Annealing，更平滑地过渡。

## 进阶诊断与综合建议

<font color='red'>在实际训练中，遇到"loss 突然变 NaN"，建议按以下流程排查：</font>

### 快速诊断清单
  - 检查学习率：
    - 先尝试将学习率除以 10，观察是否仍然发散。
  - 检查梯度范数：
    - 在崩溃前打印 torch.nn.utils.clip_grad_norm_ 的返回值，若梯度范数突然暴增，确认是梯度爆炸。
  - 检查输入数据：
    - 是否有异常值（outlier）？数据归一化是否正确？
  - 检查损失函数：
    - 是否对概率加了 eps 防零（如 log(p + 1e-8)）？
  - 检查激活函数：
    - 是否使用了容易爆炸的激活（如无界函数）？
  
### 综合解决方案

| 技术                          | 适用场景             | 原理                             |
| --------------------------- | ---------------- | ------------------------------ |
| **梯度裁剪（Gradient Clipping）** | RNN、RL、任何深度网络    | 设定梯度范数上限，直接阻止爆炸。               |
| **权重衰减（Weight Decay）**      | 通用               | 限制参数模长，间接限制激活值规模。              |
| **层归一化（LayerNorm）**         | Transformer、深层网络 | 稳定每层的激活分布，减少内部协变量偏移。           |
| **使用 AdamW 替代 Adam**        | 大模型训练            | 解耦权重衰减与学习率，优化更稳定。              |
| **混合精度检查**                  | 使用 FP16/BF16 时   | FP16 动态范围小，容易溢出；可尝试 BF16 或全精度。 |
| **Trust Region 方法**         | 强化学习（TRPO/PPO）   | 显式限制策略更新幅度，从算法层面约束步长。          |


## 一句话总结
学习率过大导致训练崩溃的本质是：优化步长超过了损失曲面的局部信任域，使参数跃入损失极高且数值不稳定的区域，进而引发梯度爆炸和数值溢出。降低学习率或采用 Warmup，本质上是给优化器戴上"缰绳"，确保它在复杂地形中稳步前行而非纵马坠崖。